# Обучение GPT на TinyStories (SageMaker Studio Lab)

Ноутбук рассчитан на **SageMaker Studio Lab** (GPU T4, до 4 ч в сутки, 15 ГБ диска), но работает и в Kaggle / Colab.

Порядок:
1. Запустить runtime с **GPU** и открыть этот ноутбук.
2. Выполнить ячейки сверху вниз.
3. Обучение идёт **в фоне**: можно закрыть вкладку, процесс доработает до конца сессии.
4. В следующую сессию выставить `RESUME = True` и запустить ячейки снова — обучение продолжится с последнего чекпоинта.

## 1. Проверка GPU

In [ ]:
!nvidia-smi

## 2. Клонирование / обновление репозитория

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/runtime57/ai360-transformer.git"
BRANCH = "main"

# на Kaggle пишем в /kaggle/working, иначе — в домашнюю папку (постоянный диск Studio Lab)
BASE_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.home()
REPO_DIR = BASE_DIR / "ai360-transformer"

if not REPO_DIR.exists():
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

## 3. Установка зависимостей

PyTorch ставится только если в окружении нет версии с поддержкой CUDA.

In [ ]:
import importlib.util, subprocess, sys

def torch_has_cuda():
    if importlib.util.find_spec("torch") is None:
        return False
    out = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                         capture_output=True, text=True)
    return out.stdout.strip() == "True"

if not torch_has_cuda():
    %pip install -q torch

%pip install -q hydra-core==1.3.7 omegaconf==2.3.1 tokenizers==0.23.2 datasets wandb tqdm requests numpy

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 4. (Опционально) Weights & Biases

Чтобы смотреть графики из браузера, поставьте `USE_WANDB = True` и введите API-ключ (https://wandb.ai/authorize).

In [ ]:
USE_WANDB = False

if USE_WANDB:
    import wandb
    wandb.login()

## 5. Параметры обучения

* `LIMIT` — сколько историй брать в train (`None` = все ~2.1M; на 15 ГБ диска и 4 ч не рекомендуется).
* `EPOCH_LEN` — шагов в одной «эпохе»; после каждой эпохи — валидация и чекпоинт.
* `RESUME = True` — продолжить с последнего чекпоинта рана `RUN_NAME`.

In [ ]:
RUN_NAME   = "tinystory_gpt"
RESUME     = False

LIMIT      = 200_000
VAL_LIMIT  = 5_000
EPOCHS     = 20
EPOCH_LEN  = 1000
BATCH_SIZE = 64      # на одну GPU; на 2×T4 общий батч = 128
SEQ_LEN    = 256
LR         = 3e-4
NUM_WORKERS = 2      # на каждую GPU; у Kaggle всего 4 CPU

SAVE_DIR = "saved"
CKPT_DIR = REPO_DIR / SAVE_DIR / RUN_NAME
LOG_FILE = REPO_DIR / f"{RUN_NAME}.log"
PID_FILE = REPO_DIR / f"{RUN_NAME}.pid"

## 6. Запуск обучения в фоне

In [ ]:
import shlex, subprocess

def latest_checkpoint(ckpt_dir):
    ckpts = sorted(Path(ckpt_dir).glob("*.pth"), key=lambda p: p.stat().st_mtime)
    return ckpts[-1].name if ckpts else None

overrides = [
    f"writer={'wandb' if USE_WANDB else 'disabled_wandb'}",
    f"writer.run_name={RUN_NAME}",
    f"datasets.train.limit={'null' if LIMIT is None else LIMIT}",
    f"datasets.val.limit={VAL_LIMIT}",
    f"datasets.train.max_seq_len={SEQ_LEN}",
    f"datasets.val.max_seq_len={SEQ_LEN}",
    f"dataloader.batch_size={BATCH_SIZE}",
    f"dataloader.num_workers={NUM_WORKERS}",
    f"optimizer.lr={LR}",
    f"trainer.epochs={EPOCHS}",
    f"trainer.epoch_len={EPOCH_LEN}",
    "trainer.save_period=1",
    f"trainer.save_dir={SAVE_DIR}",
]

if RESUME:
    ckpt = latest_checkpoint(CKPT_DIR)
    assert ckpt is not None, f"В {CKPT_DIR} нет чекпоинтов"
    print("Продолжаем с", ckpt)
    overrides.append(f"trainer.resume_from={ckpt}")
else:
    overrides.append("trainer.override=True")  # перезапишет старый ран с тем же RUN_NAME

# на нескольких GPU (Kaggle 2×T4) запускаем через torchrun — по процессу на каждую GPU (DDP)
N_GPUS = torch.cuda.device_count()
if N_GPUS > 1:
    launcher = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={N_GPUS}"]
else:
    launcher = [sys.executable]
print("GPU:", N_GPUS)

cmd = [*launcher, "train.py", "--config-name", "train_tinystory", *overrides]
print(" ".join(shlex.quote(c) for c in cmd))

log = open(LOG_FILE, "a")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=REPO_DIR,
                        start_new_session=True)  # процесс переживёт закрытие вкладки / рестарт ядра
PID_FILE.write_text(str(proc.pid))
print("PID:", proc.pid, "| лог:", LOG_FILE)

## 7. Мониторинг

Перезапускайте ячейку, чтобы увидеть свежие строки лога.

In [ ]:
import re

def is_running():
    if not PID_FILE.exists():
        return False
    pid = int(PID_FILE.read_text())
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return False
    # не считаем зомби-процесс живым
    return "Z" not in Path(f"/proc/{pid}/stat").read_text().split()[2]

text = LOG_FILE.read_text(errors="ignore") if LOG_FILE.exists() else ""
lines = [l for l in re.split(r"[\r\n]+", text) if l.strip()]
print("Статус:", "идёт обучение" if is_running() else "процесс не запущен")
print("\n".join(lines[-25:]))

In [ ]:
# метрики по эпохам
for l in lines:
    if re.match(r"\s+(epoch|loss|val_loss|grad_norm)\s+:", l):
        print(l)

## 8. Остановка обучения (при необходимости)

In [ ]:
import signal, time

STOP_FILE = CKPT_DIR / "STOP"

if is_running():
    # тренер увидит файл (раз в log_step шагов), сохранит чекпоинт на всех GPU согласованно и завершится
    STOP_FILE.touch()
    print("Создан", STOP_FILE, "— ждём сохранения чекпоинта...")
    for _ in range(300):
        if not is_running():
            break
        time.sleep(2)
    if is_running():
        os.killpg(int(PID_FILE.read_text()), signal.SIGINT)
        print("Процесс не остановился за 10 минут — отправлен SIGINT")
    else:
        print("Остановлено")
else:
    print("Процесс не запущен")

## 9. Генерация текста

Загружает `model_best.pth` и продолжает промпт (жадное декодирование).

In [ ]:
from hydra.utils import instantiate
from src.tokenizers import init_tokenizer

ckpt = torch.load(CKPT_DIR / "model_best.pth", map_location="cpu", weights_only=False)
cfg = ckpt["config"]
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = init_tokenizer(cfg.tokenizer, allow_build=False)
model = instantiate(cfg.model)
model._set_context(vocab_size=len(tokenizer.id_to_token))
model.load_state_dict(ckpt["state_dict"])
model._set_tokenizer(tokenizer)
model = model.to(device).eval()
model.max_new_tokens = 200

print("epoch:", ckpt["epoch"], "| best val_loss:", ckpt["monitor_best"])

In [ ]:
prompt = "Once upon a time, there was a little cat"

ids = [tokenizer.token_to_id["<bos>"], *tokenizer.encode(prompt, add_special_tokens=False)]
seq = torch.tensor([ids], device=device)
out = model.predict(seq)["seq"][0].tolist()
print(tokenizer.decode(out))

## 10. Скачать модель

Чекпоинты лежат в `saved/<RUN_NAME>/`. Скачать файл: правый клик по нему в файловом браузере → **Download**.

На диске Studio Lab всего 15 ГБ — удаляйте старые `checkpoint-epoch*.pth`, если место заканчивается:

In [ ]:
!du -sh {CKPT_DIR} ~/.cache/huggingface 2>/dev/null; df -h ~ | tail -1